In [1]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from frequency_encoder import FrequencyEncoder

In [2]:
# Load the data
df = pd.read_csv("data/raw/train_transaction.csv")

In [3]:
print("Loading data...")
# Sort chronologically
df = df.sort_values("TransactionDT").reset_index(drop=True)

Loading data...


In [4]:
# Create train / validation / test split
n = len(df)
train_end = int(n * 0.70)
validation_end = int(n * 0.85)
train = df.iloc[:train_end].copy()
validation = df.iloc[train_end:validation_end].copy()
test = df.iloc[validation_end:].copy()

In [5]:
# Feature engineering
def create_features(data):
    data = data.copy()
    # Time features
    data["transaction_hour"] = (
        (data["TransactionDT"] // 3600) % 24
    )
    data["transaction_day"] = (
        data["TransactionDT"] // (3600 * 24)
    )
    # Log transaction amount
    data["amount_log"] = np.log1p(
        data["TransactionAmt"]
    )
    return data
train = create_features(train)
validation = create_features(validation)
test = create_features(test)

In [6]:
# Separate target
TARGET = "isFraud"
y_train = train[TARGET]
y_validation = validation[TARGET]
y_test = test[TARGET]

In [7]:
# Remove target and ID
DROP_COLUMNS = [
    "isFraud",
    "TransactionID"
]
X_train = train.drop(columns=DROP_COLUMNS)
X_validation = validation.drop(columns=DROP_COLUMNS)
X_test = test.drop(columns=DROP_COLUMNS)

In [8]:
# Identify categorical/numerical columns
categorical_columns = X_train.select_dtypes(
    include=["object"]
).columns.tolist()
numerical_columns = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

print("\n========== FEATURES ==========")
print(
    "Numerical columns:",
    len(numerical_columns)
)
print(
    "Categorical columns:",
    len(categorical_columns)
)
print("\nCategorical columns:")
for column in categorical_columns:
    print(" -", column)


========== FEATURES ==========
Numerical columns: 381
Categorical columns: 14

Categorical columns:
 - ProductCD
 - card4
 - card6
 - P_emaildomain
 - R_emaildomain
 - M1
 - M2
 - M3
 - M4
 - M5
 - M6
 - M7
 - M8
 - M9


In [9]:
# Numerical preprocessing
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

In [10]:
LOW_CARDINALITY_THRESHOLD = 10

low_cardinality_columns = [
    column
    for column in categorical_columns
    if X_train[column].nunique(dropna=False)
    <= LOW_CARDINALITY_THRESHOLD
]

high_cardinality_columns = [
    column
    for column in categorical_columns
    if X_train[column].nunique(dropna=False)
    > LOW_CARDINALITY_THRESHOLD
]
print("\n========== CATEGORICAL ENCODING ==========")
print("\nOne-hot columns:")
for column in low_cardinality_columns:
    print(
        column,
        X_train[column].nunique(dropna=False)
    )
print("\nFrequency-encoded columns:")
for column in high_cardinality_columns:
    print(
        column,
        X_train[column].nunique(dropna=False)
    )


========== CATEGORICAL ENCODING ==========

One-hot columns:
ProductCD 5
card4 5
card6 5
M1 3
M2 3
M3 3
M4 4
M5 3
M6 3
M7 3
M8 3
M9 3

Frequency-encoded columns:
P_emaildomain 60
R_emaildomain 61


In [11]:
low_cardinality_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)
high_cardinality_pipeline = Pipeline(
    steps=[
        (
            "encoder",
            FrequencyEncoder()
        )
    ]
)

In [12]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numerical_columns
        ),
        (
            "low_cardinality",
            low_cardinality_pipeline,
            low_cardinality_columns
        ),
        (
            "high_cardinality",
            high_cardinality_pipeline,
            high_cardinality_columns
        )
    ]
)

In [13]:
# Fit ONLY on training data
print("\nFitting preprocessor on TRAIN...")
X_train_processed = preprocessor.fit_transform(
    X_train
)
print("Transforming VALIDATION...")
X_validation_processed = preprocessor.transform(
    X_validation
)
print("Transforming TEST...")
X_test_processed = preprocessor.transform(
    X_test
)


Fitting preprocessor on TRAIN...
Transforming VALIDATION...
Transforming TEST...


In [14]:
# Results
print("\n========== PROCESSED DATA ==========")
print(
    "Train:",
    X_train_processed.shape
)
print(
    "Validation:",
    X_validation_processed.shape
)
print(
    "Test:",
    X_test_processed.shape
)
print("\nPreprocessing complete!")


========== PROCESSED DATA ==========
Train: (413378, 415)
Validation: (88581, 415)
Test: (88581, 415)

Preprocessing complete!


In [15]:
import numpy as np
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
from xgboost import XGBClassifier

In [16]:
print("\n========== TARGET DISTRIBUTION ==========")
print("Train fraud rate:",
      y_train.mean())
print("Validation fraud rate:",
      y_validation.mean())
print("Test fraud rate:",
      y_test.mean())


========== TARGET DISTRIBUTION ==========
Train fraud rate: 0.03516878014795175
Validation fraud rate: 0.03434145019812375
Test fraud rate: 0.03480430340592226


In [17]:
# Calculate class imbalance
negative = (y_train == 0).sum()
positive = (y_train == 1).sum()
scale_pos_weight = negative / positive
print("\n========== CLASS IMBALANCE ==========")
print("Legitimate:", negative)
print("Fraud:", positive)
print(
    "scale_pos_weight:",
    round(scale_pos_weight, 3)
)


========== CLASS IMBALANCE ==========
Legitimate: 398840
Fraud: 14538
scale_pos_weight: 27.434


In [18]:
# XGBoost baseline
model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=scale_pos_weight,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

In [19]:
# Train
print("\n========== TRAINING ==========")
model.fit(
    X_train_processed,
    y_train,
    eval_set=[
        (X_train_processed, y_train),
        (X_validation_processed, y_validation)
    ],
    verbose=50
)


========== TRAINING ==========
[0]	validation_0-aucpr:0.33862	validation_1-aucpr:0.31663
[50]	validation_0-aucpr:0.55245	validation_1-aucpr:0.45002
[100]	validation_0-aucpr:0.60908	validation_1-aucpr:0.48115
[150]	validation_0-aucpr:0.64351	validation_1-aucpr:0.49895
[200]	validation_0-aucpr:0.67613	validation_1-aucpr:0.51123
[250]	validation_0-aucpr:0.69954	validation_1-aucpr:0.51894
[300]	validation_0-aucpr:0.72117	validation_1-aucpr:0.52764
[350]	validation_0-aucpr:0.74048	validation_1-aucpr:0.53424
[400]	validation_0-aucpr:0.75717	validation_1-aucpr:0.53728
[450]	validation_0-aucpr:0.77209	validation_1-aucpr:0.54148
[499]	validation_0-aucpr:0.78621	validation_1-aucpr:0.54533


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='aucpr', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=500,
              n_jobs=-1, num_parallel_tree=None, ...)

In [20]:
# Validation probabilities
print("\n========== VALIDATION ==========")
validation_probability = model.predict_proba(
    X_validation_processed
)[:, 1]


========== VALIDATION ==========


In [21]:
# Threshold-independent metrics
pr_auc = average_precision_score(
    y_validation,
    validation_probability
)
roc_auc = roc_auc_score(
    y_validation,
    validation_probability
)
print("PR-AUC:", round(pr_auc, 5))
print("ROC-AUC:", round(roc_auc, 5))

PR-AUC: 0.54539
ROC-AUC: 0.91642


In [22]:
# Default threshold
threshold = 0.5
validation_prediction = (
    validation_probability >= threshold
).astype(int)
precision = precision_score(
    y_validation,
    validation_prediction,
    zero_division=0
)
recall = recall_score(
    y_validation,
    validation_prediction,
    zero_division=0
)
f1 = f1_score(
    y_validation,
    validation_prediction,
    zero_division=0
)
print("\n========== THRESHOLD = 0.5 ==========")
print("Precision:", round(precision, 5))
print("Recall:", round(recall, 5))
print("F1:", round(f1, 5))
print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_validation,
        validation_prediction
    )
)


========== THRESHOLD = 0.5 ==========
Precision: 0.3388
Recall: 0.6384
F1: 0.44267

Confusion Matrix:
[[81749  3790]
 [ 1100  1942]]


In [23]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
print("\n========== THRESHOLD ANALYSIS ==========")
thresholds = np.arange(
    0.05,
    0.96,
    0.05
)
results = []
for threshold in thresholds:
    prediction = (
        validation_probability >= threshold
    ).astype(int)
    precision = precision_score(
        y_validation,
        prediction,
        zero_division=0
    )
    recall = recall_score(
        y_validation,
        prediction,
        zero_division=0
    )
    f1 = f1_score(
        y_validation,
        prediction,
        zero_division=0
    )
    tn, fp, fn, tp = confusion_matrix(
        y_validation,
        prediction
    ).ravel()
    results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp
    })
threshold_df = pd.DataFrame(results)
print(
    threshold_df.round(4).to_string(index=False)
)
best_f1 = threshold_df.loc[
    threshold_df["f1"].idxmax()
]
print("\n========== BEST F1 THRESHOLD ==========")
print(
    best_f1.to_string()
)


========== THRESHOLD ANALYSIS ==========
 threshold  precision  recall     f1  false_positives  false_negatives  true_positives
      0.05     0.0501  0.9855 0.0954            56843               44            2998
      0.10     0.0722  0.9517 0.1343            37190              147            2895
      0.15     0.0991  0.9096 0.1787            25165              275            2767
      0.20     0.1289  0.8741 0.2246            17974              383            2659
      0.25     0.1591  0.8317 0.2671            13371              512            2530
      0.30     0.1884  0.7893 0.3042            10345              641            2401
      0.35     0.2222  0.7531 0.3431             8021              751            2291
      0.40     0.2592  0.7127 0.3802             6195              874            2168
      0.45     0.2966  0.6755 0.4122             4873              987            2055
      0.50     0.3388  0.6384 0.4427             3790             1100            1942
 

In [24]:
print("\n========== COST-SENSITIVE THRESHOLD ANALYSIS ==========")
FP_COST = 1
FN_COST = 5
threshold_df["cost"] = (
    threshold_df["false_positives"] * FP_COST
    + threshold_df["false_negatives"] * FN_COST
)
best_cost = threshold_df.loc[
    threshold_df["cost"].idxmin()
]
print(
    threshold_df[
        [
            "threshold",
            "precision",
            "recall",
            "f1",
            "false_positives",
            "false_negatives",
            "cost"
        ]
    ].round(4).to_string(index=False)
)
print("\n========== LOWEST COST ==========")
print(best_cost.to_string())


========== COST-SENSITIVE THRESHOLD ANALYSIS ==========
 threshold  precision  recall     f1  false_positives  false_negatives  cost
      0.05     0.0501  0.9855 0.0954            56843               44 57063
      0.10     0.0722  0.9517 0.1343            37190              147 37925
      0.15     0.0991  0.9096 0.1787            25165              275 26540
      0.20     0.1289  0.8741 0.2246            17974              383 19889
      0.25     0.1591  0.8317 0.2671            13371              512 15931
      0.30     0.1884  0.7893 0.3042            10345              641 13550
      0.35     0.2222  0.7531 0.3431             8021              751 11776
      0.40     0.2592  0.7127 0.3802             6195              874 10565
      0.45     0.2966  0.6755 0.4122             4873              987  9808
      0.50     0.3388  0.6384 0.4427             3790             1100  9290
      0.55     0.3877  0.6029 0.4720             2896             1208  8936
      0.60     0.44

In [25]:
print("\n========== FEATURE IMPORTANCE ==========")

feature_importance = model.feature_importances_
print(
    "Number of model features:",
    len(feature_importance)
)
feature_names = preprocessor.get_feature_names_out()

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": feature_importance
})

importance_df = importance_df.sort_values(
    "importance",
    ascending=False
)

print("\n========== TOP 30 FEATURES ==========")

print(
    importance_df.head(30).to_string(index=False)
)
print("\n========== IMPORTANCE SUMMARY ==========")

print(
    importance_df["importance"].describe()
)


========== FEATURE IMPORTANCE ==========
Number of model features: 415

========== TOP 30 FEATURES ==========
                      feature  importance
                numeric__V258    0.115908
                numeric__V218    0.046719
                 numeric__V70    0.042836
                 numeric__V91    0.039025
                numeric__V294    0.035005
                numeric__V201    0.031258
                  numeric__C8    0.020846
                 numeric__V34    0.016798
                numeric__V187    0.013208
                 numeric__C14    0.011708
 low_cardinality__ProductCD_C    0.011164
                numeric__V308    0.010662
low_cardinality__card6_credit    0.009167
                numeric__V317    0.008493
                  numeric__C4    0.008382
                  numeric__C5    0.006645
                 numeric__V29    0.006523
                numeric__V283    0.006419
                numeric__V162    0.005788
 low_cardinality__ProductCD_R    0.005618
       

In [26]:
importance_df["family"] = (
    importance_df["feature"]
    .str.replace(
        r"^(numeric__|low_cardinality__|high_cardinality__)",
        "",
        regex=True
    )
    .str.extract(r"^([A-Za-z]+)")[0]
)

family_importance = (
    importance_df
    .groupby("family")["importance"]
    .sum()
    .sort_values(ascending=False)
)

print("\n========== FEATURE FAMILY IMPORTANCE ==========")

print(
    family_importance.round(4)
)


========== FEATURE FAMILY IMPORTANCE ==========
family
V                 0.7799
C                 0.0778
M                 0.0380
card              0.0291
D                 0.0287
ProductCD         0.0266
transaction       0.0047
dist              0.0026
amount            0.0025
R                 0.0023
addr              0.0023
TransactionAmt    0.0022
P                 0.0019
TransactionDT     0.0015
Name: importance, dtype: float32


In [27]:
importance_sorted = importance_df.sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

importance_sorted["cumulative_importance"] = (
    importance_sorted["importance"].cumsum()
)

print("\n========== CUMULATIVE IMPORTANCE ==========")

for n in [10, 20, 30, 50, 100, 200]:

    value = importance_sorted.loc[
        n - 1,
        "cumulative_importance"
    ]

    print(
        f"Top {n}: {value:.2%}"
    )


========== CUMULATIVE IMPORTANCE ==========
Top 10: 37.33%
Top 20: 45.22%
Top 30: 50.04%
Top 50: 56.92%
Top 100: 69.14%
Top 200: 85.11%


In [28]:
feature_names = np.asarray(
    preprocessor.get_feature_names_out()
)
v_mask = np.array([
    feature.split("__")[-1].startswith("V")
    for feature in feature_names
])

non_v_mask = ~v_mask
print("V features:", v_mask.sum())
print("Non-V features:", non_v_mask.sum())

V features: 339
Non-V features: 76


In [29]:
v_model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=scale_pos_weight,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

v_model.fit(
    X_train_processed[:, v_mask],
    y_train,
    eval_set=[
        (
            X_validation_processed[:, v_mask],
            y_validation
        )
    ],
    verbose=50
)

[0]	validation_0-aucpr:0.28125
[50]	validation_0-aucpr:0.37116
[100]	validation_0-aucpr:0.39223
[150]	validation_0-aucpr:0.40165
[200]	validation_0-aucpr:0.40641
[250]	validation_0-aucpr:0.40950
[300]	validation_0-aucpr:0.41176
[350]	validation_0-aucpr:0.41255
[400]	validation_0-aucpr:0.41365
[450]	validation_0-aucpr:0.41392
[499]	validation_0-aucpr:0.41294


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='aucpr', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=500,
              n_jobs=-1, num_parallel_tree=None, ...)

In [30]:
v_probability = v_model.predict_proba(
    X_validation_processed[:, v_mask]
)[:, 1]
v_pr_auc = average_precision_score(
    y_validation,
    v_probability
)
v_roc_auc = roc_auc_score(
    y_validation,
    v_probability
)
print("\n========== V-ONLY MODEL ==========")
print("PR-AUC:", round(v_pr_auc, 5))
print("ROC-AUC:", round(v_roc_auc, 5))


========== V-ONLY MODEL ==========
PR-AUC: 0.41211
ROC-AUC: 0.82686


In [31]:
non_v_model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=scale_pos_weight,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)
non_v_model.fit(
    X_train_processed[:, non_v_mask],
    y_train,

    eval_set=[
        (
            X_validation_processed[:, non_v_mask],
            y_validation
        )
    ],
    verbose=50
)

[0]	validation_0-aucpr:0.30136
[50]	validation_0-aucpr:0.45818
[100]	validation_0-aucpr:0.48508
[150]	validation_0-aucpr:0.49768
[200]	validation_0-aucpr:0.50784
[250]	validation_0-aucpr:0.51320
[300]	validation_0-aucpr:0.51779
[350]	validation_0-aucpr:0.52321
[400]	validation_0-aucpr:0.52712
[450]	validation_0-aucpr:0.53253
[499]	validation_0-aucpr:0.53722


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='aucpr', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=500,
              n_jobs=-1, num_parallel_tree=None, ...)

In [32]:
non_v_probability = non_v_model.predict_proba(
    X_validation_processed[:, non_v_mask]
)[:, 1]
non_v_pr_auc = average_precision_score(
    y_validation,
    non_v_probability
)
non_v_roc_auc = roc_auc_score(
    y_validation,
    non_v_probability
)
print("\n========== NON-V MODEL ==========")
print(
    "PR-AUC:",
    round(non_v_pr_auc, 5)
)
print(
    "ROC-AUC:",
    round(non_v_roc_auc, 5)
)


========== NON-V MODEL ==========
PR-AUC: 0.53728
ROC-AUC: 0.91619


In [33]:
results = pd.DataFrame([
    {
        "model": "All features",
        "features": 415,
        "PR_AUC": 0.54539,
        "ROC_AUC": 0.91642
    },
    {
        "model": "V-only",
        "features": int(v_mask.sum()),
        "PR_AUC": 0.41211,
        "ROC_AUC": 0.82686
    },
    {
        "model": "Non-V",
        "features": int(non_v_mask.sum()),
        "PR_AUC": 0.53728,
        "ROC_AUC": 0.91619
    }
])

print(results)

          model  features   PR_AUC  ROC_AUC
0  All features       415  0.54539  0.91642
1        V-only       339  0.41211  0.82686
2         Non-V        76  0.53728  0.91619


In [34]:
v_importance = importance_df[
    importance_df["feature"].str.contains(
        r"__V\d+$",
        regex=True
    )
].sort_values(
    "importance",
    ascending=False
)

top_v_features = v_importance.head(30)["feature"].values

top_v_mask = np.isin(
    feature_names,
    top_v_features
)

print("Top V features:", top_v_mask.sum())

Top V features: 30


In [35]:
X_train_top_v = X_train_processed[
    :,
    top_v_mask
]

X_validation_top_v = X_validation_processed[
    :,
    top_v_mask
]

In [36]:
non_v_plus_top_v_mask = (
    non_v_mask | top_v_mask
)

print(
    "Features:",
    non_v_plus_top_v_mask.sum()
)

Features: 106


In [37]:
# NON-V + TOP 30 V FEATURES
# ==========================================
# NON-V + TOP 30 V FEATURES
# ==========================================

feature_names = np.asarray(
    preprocessor.get_feature_names_out()
)

# V features from the full-model importance
v_importance = importance_df[
    importance_df["feature"].str.contains(
        r"__V\d+$",
        regex=True
    )
].sort_values(
    "importance",
    ascending=False
)

top_v_features = (
    v_importance
    .head(30)["feature"]
    .values
)

top_v_mask = np.isin(
    feature_names,
    top_v_features
)

# Everything except V
non_v_mask = ~np.array([
    feature.split("__")[-1].startswith("V")
    for feature in feature_names
])

# Combine Non-V + Top 30 V
non_v_plus_top_v_mask = (
    non_v_mask | top_v_mask
)

print("\n========== FEATURE SELECTION ==========")

print("Total features:", len(feature_names))
print("Non-V features:", non_v_mask.sum())
print("Top V features:", top_v_mask.sum())
print(
    "Final features:",
    non_v_plus_top_v_mask.sum()
)

print("\n========== TOP 30 V FEATURES ==========")

print(
    "\n".join(top_v_features)
)


========== FEATURE SELECTION ==========
Total features: 415
Non-V features: 76
Top V features: 30
Final features: 106

========== TOP 30 V FEATURES ==========
numeric__V258
numeric__V218
numeric__V70
numeric__V91
numeric__V294
numeric__V201
numeric__V34
numeric__V187
numeric__V308
numeric__V317
numeric__V29
numeric__V283
numeric__V162
numeric__V243
numeric__V128
numeric__V95
numeric__V90
numeric__V102
numeric__V312
numeric__V45
numeric__V296
numeric__V303
numeric__V330
numeric__V163
numeric__V69
numeric__V311
numeric__V156
numeric__V64
numeric__V282
numeric__V149


In [38]:
X_train_reduced = X_train_processed[
    :,
    non_v_plus_top_v_mask
]

X_validation_reduced = X_validation_processed[
    :,
    non_v_plus_top_v_mask
]

print("\n========== REDUCED DATA ==========")

print(
    "Train:",
    X_train_reduced.shape
)

print(
    "Validation:",
    X_validation_reduced.shape
)


========== REDUCED DATA ==========
Train: (413378, 106)
Validation: (88581, 106)


In [39]:
reduced_model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,

    subsample=0.8,
    colsample_bytree=0.8,

    objective="binary:logistic",
    eval_metric="aucpr",

    scale_pos_weight=scale_pos_weight,

    tree_method="hist",

    random_state=42,
    n_jobs=-1
)

print("\n========== TRAINING NON-V + TOP 30 V ==========")

reduced_model.fit(
    X_train_reduced,
    y_train,

    eval_set=[
        (
            X_validation_reduced,
            y_validation
        )
    ],

    verbose=50
)


========== TRAINING NON-V + TOP 30 V ==========
[0]	validation_0-aucpr:0.33835
[50]	validation_0-aucpr:0.44862
[100]	validation_0-aucpr:0.48641
[150]	validation_0-aucpr:0.50088
[200]	validation_0-aucpr:0.51114
[250]	validation_0-aucpr:0.51928
[300]	validation_0-aucpr:0.52507
[350]	validation_0-aucpr:0.53035
[400]	validation_0-aucpr:0.53612
[450]	validation_0-aucpr:0.54078
[499]	validation_0-aucpr:0.54473


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='aucpr', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=500,
              n_jobs=-1, num_parallel_tree=None, ...)

In [40]:
reduced_probability = reduced_model.predict_proba(
    X_validation_reduced
)[:, 1]

reduced_pr_auc = average_precision_score(
    y_validation,
    reduced_probability
)

reduced_roc_auc = roc_auc_score(
    y_validation,
    reduced_probability
)

print("\n========== NON-V + TOP 30 V ==========")

print(
    "PR-AUC:",
    round(reduced_pr_auc, 5)
)

print(
    "ROC-AUC:",
    round(reduced_roc_auc, 5)
)


========== NON-V + TOP 30 V ==========
PR-AUC: 0.54478
ROC-AUC: 0.91612


In [41]:
comparison = pd.DataFrame([
    {
        "model": "All features",
        "features": 415,
        "PR-AUC": 0.54539,
        "ROC-AUC": 0.91642
    },
    {
        "model": "Non-V",
        "features": int(non_v_mask.sum()),
        "PR-AUC": 0.53728,
        "ROC-AUC": 0.91619
    },
    {
        "model": "Non-V + Top 30 V",
        "features": int(non_v_plus_top_v_mask.sum()),
        "PR-AUC": reduced_pr_auc,
        "ROC-AUC": reduced_roc_auc
    }
])

print("\n========== MODEL COMPARISON ==========")

print(
    comparison.round(5).to_string(index=False)
)


========== MODEL COMPARISON ==========
           model  features  PR-AUC  ROC-AUC
    All features       415 0.54539  0.91642
           Non-V        76 0.53728  0.91619
Non-V + Top 30 V       106 0.54478  0.91612


In [42]:
thresholds = np.arange(
    0.05,
    0.96,
    0.05
)

reduced_results = []

for threshold in thresholds:

    prediction = (
        reduced_probability >= threshold
    ).astype(int)

    precision = precision_score(
        y_validation,
        prediction,
        zero_division=0
    )

    recall = recall_score(
        y_validation,
        prediction,
        zero_division=0
    )

    f1 = f1_score(
        y_validation,
        prediction,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_validation,
        prediction
    ).ravel()

    cost = (
        fp * 1
        + fn * 5
    )

    reduced_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp,
        "cost": cost
    })


reduced_threshold_df = pd.DataFrame(
    reduced_results
)

print(
    reduced_threshold_df.round(4).to_string(
        index=False
    )
)

 threshold  precision  recall     f1  false_positives  false_negatives  true_positives  cost
      0.05     0.0483  0.9859 0.0920            59149               43            2999 59364
      0.10     0.0685  0.9569 0.1278            39616              131            2911 40271
      0.15     0.0925  0.9247 0.1682            27598              229            2813 28743
      0.20     0.1180  0.8843 0.2082            20111              352            2690 21871
      0.25     0.1444  0.8445 0.2466            15225              473            2569 17590
      0.30     0.1728  0.8189 0.2854            11922              551            2491 14677
      0.35     0.2021  0.7860 0.3216             9438              651            2391 12693
      0.40     0.2346  0.7525 0.3577             7467              753            2289 11232
      0.45     0.2702  0.7176 0.3926             5895              859            2183 10190
      0.50     0.3043  0.6719 0.4189             4673              998

In [43]:
best_reduced_cost = reduced_threshold_df.loc[
    reduced_threshold_df["cost"].idxmin()
]

print("\n========== BEST REDUCED-MODEL THRESHOLD ==========")

print(
    best_reduced_cost.to_string()
)


========== BEST REDUCED-MODEL THRESHOLD ==========
threshold             0.700000
precision             0.498428
recall                0.521039
f1                    0.509482
false_positives    1595.000000
false_negatives    1457.000000
true_positives     1585.000000
cost               8880.000000


In [44]:
# ==========================================
# FINAL TEST EVALUATION
# ==========================================

print("\n========== FINAL TEST EVALUATION ==========")

# Test features using the already-selected
# 106-feature mask
X_test_reduced = X_test_processed[
    :,
    non_v_plus_top_v_mask
]

# Test probabilities
test_probability = reduced_model.predict_proba(
    X_test_reduced
)[:, 1]

# Threshold was selected on validation only
FINAL_THRESHOLD = 0.70

test_prediction = (
    test_probability >= FINAL_THRESHOLD
).astype(int)


# ------------------------------------------
# Threshold-independent metrics
# ------------------------------------------

test_pr_auc = average_precision_score(
    y_test,
    test_probability
)

test_roc_auc = roc_auc_score(
    y_test,
    test_probability
)


# ------------------------------------------
# Threshold-dependent metrics
# ------------------------------------------

test_precision = precision_score(
    y_test,
    test_prediction,
    zero_division=0
)

test_recall = recall_score(
    y_test,
    test_prediction,
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    test_prediction,
    zero_division=0
)


# ------------------------------------------
# Confusion matrix
# ------------------------------------------

tn, fp, fn, tp = confusion_matrix(
    y_test,
    test_prediction
).ravel()


# ------------------------------------------
# Cost
# ------------------------------------------

FP_COST = 1
FN_COST = 5

test_cost = (
    fp * FP_COST
    + fn * FN_COST
)


# ------------------------------------------
# Print results
# ------------------------------------------

print("\nPR-AUC:", round(test_pr_auc, 5))
print("ROC-AUC:", round(test_roc_auc, 5))

print("\nThreshold:", FINAL_THRESHOLD)

print(
    "Precision:",
    round(test_precision, 5)
)

print(
    "Recall:",
    round(test_recall, 5)
)

print(
    "F1:",
    round(test_f1, 5)
)

print("\n========== CONFUSION MATRIX ==========")

print(
    confusion_matrix(
        y_test,
        test_prediction
    )
)

print("\n========== CONFUSION MATRIX VALUES ==========")

print("True negatives :", tn)
print("False positives:", fp)
print("False negatives:", fn)
print("True positives :", tp)

print("\n========== COST ==========")

print("FP cost:", FP_COST)
print("FN cost:", FN_COST)
print("Total cost:", test_cost)


========== FINAL TEST EVALUATION ==========

PR-AUC: 0.49387
ROC-AUC: 0.89977

Threshold: 0.7
Precision: 0.44161
Recall: 0.50049
F1: 0.46921

========== CONFUSION MATRIX ==========
[[83547  1951]
 [ 1540  1543]]

========== CONFUSION MATRIX VALUES ==========
True negatives : 83547
False positives: 1951
False negatives: 1540
True positives : 1543

========== COST ==========
FP cost: 1
FN cost: 5
Total cost: 9651


In [45]:
import numpy as np
import pandas as pd

FINAL_THRESHOLD = 0.70


def get_risk_level(probability):
    if probability >= FINAL_THRESHOLD:
        return "HIGH"
    elif probability >= 0.30:
        return "MEDIUM"
    else:
        return "LOW"


def get_risk_score(probability):
    """
    Convert model probability to a 0-100 risk score.
    """
    return round(float(probability) * 100, 2)


def get_recommended_action(probability):
    if probability >= FINAL_THRESHOLD:
        return "BLOCK / REVIEW"
    elif probability >= 0.30:
        return "VERIFY"
    else:
        return "APPROVE"

In [46]:
transaction_index = 0

probability = test_probability[transaction_index]

risk_score = get_risk_score(probability)
risk_level = get_risk_level(probability)
action = get_recommended_action(probability)

print("\n========== RISK ASSESSMENT ==========")

print("Fraud probability:", round(probability, 4))
print("Risk score:", risk_score, "/ 100")
print("Risk level:", risk_level)
print("Recommended action:", action)


========== RISK ASSESSMENT ==========
Fraud probability: 0.0039
Risk score: 0.39 / 100
Risk level: LOW
Recommended action: APPROVE


In [47]:
def assess_transaction(
    transaction_index,
    transactions,
    test_probability
):
    
    row = transactions.iloc[transaction_index]

    probability = float(
        test_probability[transaction_index]
    )

    result = {
        "transaction_id": int(row["TransactionID"]),
        "amount": float(row["TransactionAmt"]),
        "product": row["ProductCD"],
        "card_type": row["card4"],
        "card_category": row["card6"],

        "fraud_probability": round(
            probability,
            4
        ),

        "risk_score": get_risk_score(
            probability
        ),

        "risk_level": get_risk_level(
            probability
        ),

        "recommended_action":
            get_recommended_action(
                probability
            )
    }

    return result

In [48]:
result = assess_transaction(
    transaction_index=0,
    transactions=test,
    test_probability=test_probability
)

print("\n========== TRANSACTION RISK ==========")

for key, value in result.items():
    print(f"{key}: {value}")


========== TRANSACTION RISK ==========
transaction_id: 3488959
amount: 57.95
product: W
card_type: mastercard
card_category: debit
fraud_probability: 0.0039
risk_score: 0.39
risk_level: LOW
recommended_action: APPROVE


In [49]:
import shap
explainer = shap.TreeExplainer(
    reduced_model
)
transaction_shap = explainer.shap_values(
    X_test_reduced[transaction_index].reshape(1, -1)
)
selected_feature_names = feature_names[
    non_v_plus_top_v_mask
]

In [50]:
explanation_df = pd.DataFrame({
    "feature": selected_feature_names,
    "shap_value": transaction_shap.flatten()
})

explanation_df["abs_shap"] = (
    explanation_df["shap_value"]
    .abs()
)

explanation_df = explanation_df.sort_values(
    "abs_shap",
    ascending=False
)

print("\n========== TOP RISK SIGNALS ==========")

print(
    explanation_df.head(10)[
        ["feature", "shap_value"]
    ].to_string(index=False)
)


========== TOP RISK SIGNALS ==========
       feature  shap_value
  numeric__C13   -0.898249
  numeric__C14   -0.533870
   numeric__C1    0.482636
  numeric__V70   -0.415897
numeric__card1   -0.412117
   numeric__C5   -0.393399
  numeric__D10   -0.367388
numeric__card5   -0.283375
numeric__addr1   -0.276881
   numeric__D2   -0.252071


In [51]:
def explain_feature(feature):

    if "__V" in feature:
        return "Transaction verification signal"

    if "__C" in feature:
        return "Transaction count / activity signal"

    if "__D" in feature:
        return "Transaction timing signal"

    if "__M" in feature:
        return "Match / verification signal"

    if "card" in feature.lower():
        return "Payment card signal"

    if "ProductCD" in feature:
        return "Product category signal"

    if "email" in feature.lower():
        return "Email-domain signal"

    if "addr" in feature.lower():
        return "Billing/address signal"

    if "dist" in feature.lower():
        return "Distance signal"

    if "TransactionAmt" in feature:
        return "Transaction amount signal"

    if "TransactionDT" in feature:
        return "Transaction timing signal"

    return "Transaction behavior signal"

In [52]:
top_signals = explanation_df.head(5).copy()

top_signals["signal"] = (
    top_signals["feature"]
    .apply(explain_feature)
)

print("\n========== HUMAN-READABLE SIGNALS ==========")

print(
    top_signals[
        ["signal", "shap_value"]
    ].to_string(index=False)
)


========== HUMAN-READABLE SIGNALS ==========
                             signal  shap_value
Transaction count / activity signal   -0.898249
Transaction count / activity signal   -0.533870
Transaction count / activity signal    0.482636
    Transaction verification signal   -0.415897
                Payment card signal   -0.412117


In [53]:
def get_explanation_tables(explanation_df, n=5):

    positive = (
        explanation_df[
            explanation_df["shap_value"] > 0
        ]
        .sort_values(
            "shap_value",
            ascending=False
        )
        .head(n)
        .copy()
    )

    negative = (
        explanation_df[
            explanation_df["shap_value"] < 0
        ]
        .sort_values(
            "shap_value",
            ascending=True
        )
        .head(n)
        .copy()
    )

    positive["direction"] = "INCREASED RISK"
    negative["direction"] = "REDUCED RISK"

    return positive, negative

In [54]:
positive_signals, negative_signals = (
    get_explanation_tables(
        explanation_df,
        n=5
    )
)

print("\n========== RISK-INCREASING SIGNALS ==========")

print(
    positive_signals[
        ["feature", "shap_value"]
    ].to_string(index=False)
)

print("\n========== RISK-REDUCING SIGNALS ==========")

print(
    negative_signals[
        ["feature", "shap_value"]
    ].to_string(index=False)
)


========== RISK-INCREASING SIGNALS ==========
               feature  shap_value
           numeric__C1    0.482636
           numeric__C9    0.171213
           numeric__C2    0.088860
numeric__TransactionDT    0.049220
 low_cardinality__M6_F    0.048356

========== RISK-REDUCING SIGNALS ==========
       feature  shap_value
  numeric__C13   -0.898249
  numeric__C14   -0.533870
  numeric__V70   -0.415897
numeric__card1   -0.412117
   numeric__C5   -0.393399


In [55]:
probability = test_probability[0]

print("========== TRANSACTION 0 RISK ==========")
print("Fraud probability:", round(probability, 4))
print("Risk score:", round(probability * 100, 2), "/ 100")
print("Threshold:", 0.70)

========== TRANSACTION 0 RISK ==========
Fraud probability: 0.0039
Risk score: 0.39 / 100
Threshold: 0.7


In [56]:
high_risk_indices = np.where(
    test_probability >= 0.70
)[0]

print(
    "High-risk transactions:",
    len(high_risk_indices)
)

print(
    "First 10 high-risk indices:",
    high_risk_indices[:10]
)

High-risk transactions: 3494
First 10 high-risk indices: [  6  46 109 110 111 112 241 267 320 330]


In [57]:
high_risk_index = high_risk_indices[0]

probability = test_probability[high_risk_index]

print("\n========== HIGH-RISK TRANSACTION ==========")

print("Test index:", high_risk_index)
print("Fraud probability:", round(probability, 4))
print("Risk score:", round(probability * 100, 2), "/ 100")
print("Threshold:", 0.70)

if probability >= 0.70:
    print("Risk level: HIGH")
    print("Recommended action: REVIEW / BLOCK")


========== HIGH-RISK TRANSACTION ==========
Test index: 6
Fraud probability: 0.72
Risk score: 72.0 / 100
Threshold: 0.7
Risk level: HIGH
Recommended action: REVIEW / BLOCK


In [58]:
high_risk_shap = explainer.shap_values(
    X_test_reduced[high_risk_index].reshape(1, -1)
)

high_risk_explanation = pd.DataFrame({
    "feature": selected_feature_names,
    "shap_value": high_risk_shap.reshape(-1)
})

high_risk_explanation["abs_shap"] = (
    high_risk_explanation["shap_value"].abs()
)

high_risk_explanation = (
    high_risk_explanation
    .sort_values("abs_shap", ascending=False)
)

print("\n========== HIGH-RISK SIGNALS ==========")

print(
    high_risk_explanation.head(10)[
        ["feature", "shap_value"]
    ].to_string(index=False)
)


========== HIGH-RISK SIGNALS ==========
                      feature  shap_value
                 numeric__C14    1.003362
                 numeric__C13    0.615489
       numeric__TransactionDT   -0.267542
low_cardinality__card6_credit   -0.179907
                  numeric__C1   -0.177932
                numeric__V258   -0.152951
                 numeric__C11   -0.130934
 low_cardinality__ProductCD_C    0.116790
                 numeric__D15   -0.115258
                  numeric__C5    0.107557


In [59]:
import joblib
from pathlib import Path

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(
    reduced_model,
    MODEL_DIR / "razorshield_model.pkl"
)

joblib.dump(
    preprocessor,
    MODEL_DIR / "razorshield_preprocessor.pkl"
)

joblib.dump(
    selected_feature_names,
    MODEL_DIR / "razorshield_features.pkl"
)

print("========== MODEL SAVED ==========")
print("Model:", MODEL_DIR / "razorshield_model.pkl")
print("Preprocessor:", MODEL_DIR / "razorshield_preprocessor.pkl")
print("Features:", MODEL_DIR / "razorshield_features.pkl")
print("Number of selected features:", len(selected_feature_names))

========== MODEL SAVED ==========
Model: models\razorshield_model.pkl
Preprocessor: models\razorshield_preprocessor.pkl
Features: models\razorshield_features.pkl
Number of selected features: 106


In [61]:
import sklearn
print(sklearn.__version__)

1.6.1


In [62]:
print(type(test_probability))
print(len(test_probability))

<class 'numpy.ndarray'>
88581


In [64]:
joblib.dump(
    test_probability,
     MODEL_DIR /"razorshield_test_predictions.pkl"
)

print("Saved successfully!")

Saved successfully!


In [65]:
joblib.dump(
    test,
    MODEL_DIR /"razorshield_test_transactions.pkl"
)

print("Test transactions saved!")

Test transactions saved!
